# IDW + ST-GNN correction: manual training and evaluation

Train either **PM2.5** or **NO₂** from the frozen **ds-v0.3.0** snapshot. Concentrations are in **µg/m³**.
This notebook calls the maintained Python implementation; it does not contain a second model implementation.

| Target | Fixed held-out split | LOSO per station group |
|---|---|---|
| PM2.5 | Supported | Supported |
| NO₂ | Not supported | Supported |

Run the numbered sections in order. Change settings before training, inspect validation results,
then enable final test evaluation only after your choices are frozen. Do not tune on test results.
Nothing is registered or promoted with the default settings.

## 1. Prepare a Python environment and open this notebook

From the repository root, use a dedicated Python 3.11 environment. Run these commands in a terminal,
**not as notebook cells** (Windows PowerShell example):

```powershell
python -m venv .venv-spatial
.\.venv-spatial\Scripts\python.exe -m pip install -r src/data_worker/requirements.txt
.\.venv-spatial\Scripts\python.exe -m pip install jupyterlab ipykernel
.\.venv-spatial\Scripts\python.exe -m ipykernel install --user --name sippolusi-spatial --display-name "SIPPolusi Spatial"
.\.venv-spatial\Scripts\python.exe -m jupyterlab notebooks/spatial_model
```

Select **SIPPolusi Spatial** as the kernel. On macOS/Linux use `.venv-spatial/bin/python`.
A CPU kernel works for both models. ST-GNN CUDA runs require a compatible PyTorch/CUDA environment;
change `DEVICE` only after confirming `torch.cuda.is_available()`.

LightGBM is pinned to 4.6.0 in the project requirements. Other dependencies are not fully locked:
preserve the output bundle's recorded versions and the same hardware when reproducing a run.
If Windows reports duplicate OpenMP/DLL errors, restart the kernel and use a clean environment;
do not suppress the runtime's integrity checks. The setup cell loads NumPy before PyTorch,
which avoids the observed import conflict in the current development environment.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import sys
import json
import os
import importlib.metadata

# Set an absolute repository path if you launched Jupyter outside the checkout.
REPO_ROOT = None  # Example: Path(r"D:/Github/SIPPolusi")
if REPO_ROOT is None:
    REPO_ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
                      if (p / "src/data_worker/spatial_model").is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Set REPO_ROOT to the SIPPolusi repository and rerun this cell.")
ROOT = Path(REPO_ROOT).resolve()
sys.path.insert(0, str(ROOT / "src/data_worker"))

import numpy as np
import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from IPython import get_ipython
if get_ipython() is not None:
    get_ipython().run_line_magic("matplotlib", "inline")
from spatial_model.data.validation import verify_snapshot
from spatial_model.configs import load_model_config
print("Repository:", ROOT)
print("Python:", sys.version.split()[0], "Kernel:", sys.executable)

## 2. Choose target, protocol, and run settings

- `TARGET`: `"pm25"` or `"no2"`. NO₂ requires `PROTOCOL = "loso"`.
- `PROTOCOL`: `"fixed_split"` or `"loso"`. Fixed split holds out both station groups and time blocks.
- `FOLDS`: a list such as `[0]` or `[0, 1]`, or `"all"`. Ignored for fixed split.
  LOSO holds out entire related station groups, not individual rows.
- `RUN_ROOT`: a new folder per experiment. To inspect an old experiment, supply its existing path
  and set `RUN_TRAINING = False`. Do not change its target/protocol/folds.
- `EVALUATE_TEST`: leave false while developing; set true for the final report after freezing choices.

ST-GNN defaults to 100 epochs with validation early stopping. Set `EPOCHS = 1` for a workflow smoke
test, not a benchmark. CPU LOSO across all PM2.5 folds may take a long time. Full LOSO is explicit,
never the default. Re-running the settings cell creates a fresh timestamped run unless you set a path.

In [ ]:
MODEL_FAMILY = "stgnn"
TARGET = "pm25"
PROTOCOL = "fixed_split"
FOLDS = [0]  # Or [0, 1], or "all"; only used for LOSO.
SEED = 42
SNAPSHOT = ROOT / "docs/experiments/ds-v0.3.0"
RUN_ROOT = ROOT / "checkpoints/spatial_model" / MODEL_FAMILY / TARGET / PROTOCOL / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_TRAINING = True
RUN_EVALUATION = True
EVALUATE_TEST = False

# ST-GNN settings (ignored by LightGBM).
EPOCHS = 100
DEVICE = "cpu"
BATCH_SIZE = 8
THREADS = 2
RESUME = False  # ST-GNN only: use the same RUN_ROOT after an interrupted run.

# Publishing always requires separate explicit opt-in below.
REGISTER_MODEL = False
PROMOTE_MODEL = False
BUNDLE_TO_REGISTER = None  # Set an explicit canonical production bundle path.

## 3. Verify inputs and inspect the planned folds

This verifies all frozen snapshot checksums before fitting. The fold table shows label counts,
including groups without final test labels. There are 103 defined PM2.5 groups and 14 NO₂ groups;
the number of evaluable folds can be smaller. Missing test labels are reported, not scored as zero.

Sensor inputs exclude the held group throughout LOSO; fixed-split sensor inputs use training stations.
ST-GNN selects checkpoints using validation on non-held groups in LOSO; LightGBM's existing
configuration was selected using held-group validation. Their validation scores therefore have
different meanings. Final held-group test predictions are the basis for comparison.

In [ ]:
if TARGET not in ("pm25", "no2") or PROTOCOL not in ("fixed_split", "loso"):
    raise ValueError("Choose a supported TARGET and PROTOCOL.")
if TARGET == "no2" and PROTOCOL != "loso":
    raise ValueError("NO2 supports LOSO only; set PROTOCOL = 'loso'.")
if SEED != 42:
    raise ValueError("Use seed 42 for the maintained validation-selected configuration.")
if MODEL_FAMILY == "lightgbm" and RESUME:
    raise ValueError("LightGBM has no checkpoint resume; use a new run folder.")
os.environ["THREADS"] = str(THREADS)
hashes = verify_snapshot(SNAPSHOT)
print("Snapshot verified:", len(hashes), "files")
versions = {name: importlib.metadata.version(name) for name in
            ("numpy", "pandas", "torch", "torch-geometric", "lightgbm", "pyarrow")}
display(pd.Series(versions, name="installed version").to_frame())
display(load_model_config(MODEL_FAMILY))

fold_table = pd.read_parquet(SNAPSHOT / "splits/loso_folds.parquet")
fold_table = fold_table[fold_table.target == TARGET][["fold", "group_id"]].drop_duplicates().sort_values("fold")
labels = pd.read_parquet(SNAPSHOT / "labels.parquet")
counts = labels[labels[f"{TARGET}_ugm3"].notna()].groupby(["group_id", "period"]).size().unstack(fill_value=0)
fold_table = fold_table.join(counts, on="group_id").fillna(0)
display(fold_table)
available = fold_table.fold.astype(int).tolist()
if PROTOCOL == "fixed_split":
    selected_folds = [None]
else:
    selected_folds = available if FOLDS == "all" else list(FOLDS)
    if not selected_folds or len(set(selected_folds)) != len(selected_folds):
        raise ValueError("Choose at least one unique fold.")
    if any(fold not in available for fold in selected_folds):
        raise ValueError(f"Unknown fold; choose from {available}")
print("Selected folds:", selected_folds)
print("Run folder:", RUN_ROOT)
if RUN_TRAINING and RUN_ROOT.exists() and not RESUME:
    raise FileExistsError("Use a new RUN_ROOT, or set RUN_TRAINING=False to inspect an existing run.")
if not RUN_TRAINING and not RUN_ROOT.is_dir():
    raise FileNotFoundError("Set RUN_ROOT to an existing experiment when RUN_TRAINING=False.")

## 4. Train using the shared stgnn implementation

Each fold gets its own checkpoint and preprocessing, with validation results saved during training.
Training here never evaluates the final test block. For ST-GNN, validation may select epoch zero:
that means the selected model is plain IDW with zero correction, not evidence of a learned improvement.
Resume is for interrupted ST-GNN runs: keep the same settings, snapshot and RUN_ROOT and set
`RESUME = True`. Completed folds in a multi-fold run may be resumed without restarting from scratch.

LightGBM reuses the maintained full-feature configuration. Applying it to PM2.5 is a reproducible
comparison run, not a claim that the parameters were newly optimized for that target.

In [ ]:
from spatial_model.models.stgnn.train import train

run_paths = {fold: RUN_ROOT / ("fixed-split" if fold is None else f"fold-{fold}")
             for fold in selected_folds}
if RUN_TRAINING:
    RUN_ROOT.mkdir(parents=True, exist_ok=RESUME)
    for fold, output in run_paths.items():
        print("Training:", TARGET, PROTOCOL, "fold", fold, "->", output)
        fold_resume = RESUME and (output / "progress.pt").is_file()
        if RESUME and output.exists() and not fold_resume:
            raise FileExistsError(f"Incomplete fold without progress.pt: {output}. Choose a fresh run or inspect the directory.")
        bundle = train(SNAPSHOT, output, target=TARGET, protocol=PROTOCOL, fold=fold, seed=SEED, epochs=EPOCHS, device=DEVICE, batch_size=BATCH_SIZE, threads=THREADS, resume=fold_resume, evaluate_test=False)
        print("Saved bundle:", bundle)
else:
    for fold, output in run_paths.items():
        manifest_path = output / "bundle/bundle.json"
        if not manifest_path.is_file():
            print(f"Unfinished fold {fold}: no completed bundle; retaining it in coverage.")
            continue
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        metadata = manifest["metadata"]
        recorded_protocol = metadata.get("protocol", "loso" if metadata.get("fold") is not None else "serving")
        recorded_family = manifest.get("model_family", "stgnn" if manifest["target"] == "pm25" else "lightgbm")
        if (manifest["target"], recorded_family, recorded_protocol, metadata.get("fold")) != (TARGET, MODEL_FAMILY, PROTOCOL, fold):
            raise ValueError(f"Saved checkpoint does not match notebook settings: {output}")
    print("Training skipped; using matching existing checkpoints.")

## 5. Review validation results and training history

Inspect the pooled and station-level validation tables, label counts, checkpoint metadata, and
(for ST-GNN) history before final testing. Validation figures are not directly comparable across
the two model families' LOSO policies. Bundles record snapshot/config/source hashes and versions.
A missing artifact is shown explicitly; a failed training run must be resolved before evaluation.

In [ ]:
for fold, output in run_paths.items():
    print("Fold:", fold, "Artifacts:", output)
    manifest_path = output / "bundle/bundle.json"
    if not manifest_path.exists():
        print("No completed bundle: finish or resume this run first.")
        continue
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    display(pd.Series(manifest["metadata"], name="checkpoint metadata").to_frame())
    for filename in ("idw_comparison.csv", "evaluation_metrics.csv"):
        path = output / filename
        if path.exists():
            display(pd.read_csv(path))
    history_path = output / "history.csv"
    if history_path.exists():
        history = pd.read_csv(history_path)
        if not history.empty:
            history.plot(x="epoch", y="validation_rmse", title=f"Fold {fold}: validation RMSE (µg/m³)", grid=True)
            plt.show()

## 6. Evaluate saved checkpoints, including final held-out testing

Set `EVALUATE_TEST = True` in settings only when choices are frozen, then rerun settings and the
preflight/train cells with `RUN_TRAINING = False` and your saved RUN_ROOT. Alternatively change
the flag in this cell before running it. Evaluation reloads saved weights/preprocessing without
fitting. It verifies snapshot identity and sensor policy. It writes predictions and provenance under
each run's `evaluation/` folder; existing reports there are overwritten for the requested splits.

LOSO test rows belong to the held group in the final temporal test block. No test labels means
an unevaluable split. The snapshot and split rules remain unchanged.

During inspection, unfinished folds without a completed bundle are skipped by evaluation
and remain in the coverage table. Existing bundles must still match your settings.

In [ ]:
from spatial_model.evaluation.stgnn.evaluate import evaluate

evaluation_splits = ("val", "test") if EVALUATE_TEST else ("val",)
print("Evaluating splits:", evaluation_splits)
if RUN_EVALUATION:
    for fold, output in run_paths.items():
        if not (output / "bundle/bundle.json").exists():
            print(f"Skipping unfinished fold {fold}: no completed checkpoint at {output}")
            continue
        evaluate(SNAPSHOT, output / "best.pt", output / "evaluation", splits=evaluation_splits, device=DEVICE, batch_size=BATCH_SIZE, threads=THREADS)
else:
    print("Evaluation skipped; the summary reads existing evaluation reports.")

## 7. Inspect fold coverage and pooled prediction metrics

Pooled RMSE/MAE/R² are calculated from concatenated prediction rows, **not averages of fold RMSE**.
Check coverage before interpreting them: selected-fold LOSO is a subset experiment, not full LOSO.
ST-GNN includes a paired IDW baseline. Missing and unevaluable splits are not silently counted as
successful folds. Stale or unverified reports are excluded; rerun evaluation to refresh them.
A defined fold without test labels is still listed in coverage. ST-GNN LOSO validation repeats
non-held observations across folds; its pooled validation is not an out-of-fold estimate.

Lower RMSE/MAE is better; negative R² is possible; bias is predicted minus observed.
Sensor-station results do not establish accuracy everywhere on the 100 m grid. NO₂ sensor-network
geometry/bias and the limited reference network constrain interpretation. The notebook does not
calibrate uncertainty or demonstrate deployment readiness.

The example plot uses only a report marked `available` for the plotted split.
Coverage is saved even when no selected fold has an available report.

In [ ]:
from spatial_model.training.results import summarize

summary, coverage = summarize(RUN_ROOT, selected_folds, splits=evaluation_splits)
display(coverage)
display(summary)
if PROTOCOL == "loso":
    print(f"Selected {len(selected_folds)} / {len(available)} defined folds.")
    print("All defined folds selected:", set(selected_folds) == set(available))
coverage.to_csv(RUN_ROOT / "coverage.csv", index=False)
if not summary.empty:
    summary.to_csv(RUN_ROOT / "summary.csv", index=False)
    plot_split = evaluation_splits[-1]
    plot_folds = coverage.loc[(coverage.split == plot_split) & (coverage.status == "available"), "fold"]
    first = next((run_paths[None if pd.isna(fold) else int(fold)] / "evaluation" / f"{plot_split}_predictions.parquet"
                  for fold in plot_folds), None)
    if first is not None:
        predictions = pd.read_parquet(first)
        display(predictions.head(10))
        ax = predictions.plot.scatter(x="observed", y="predicted", alpha=0.4,
                                      title="Example fold: observed vs predicted (µg/m³)")
        bounds = [min(predictions.observed.min(), predictions.predicted.min()),
                  max(predictions.observed.max(), predictions.predicted.max())]
        ax.plot(bounds, bounds, color="black", linestyle="--")
        plt.show()
else:
    print("No prediction reports found for the requested splits; run evaluation first.")

## 8. Optional registration and promotion — disabled by default

Research comparisons do not change the production pairing (**PM2.5 ST-GNN**, **NO₂ LightGBM**).
Alternative target/model bundles cannot be served by the current production pipeline.
Never promote a held-out fold checkpoint as an all-station serving fit.

To publish a reviewed canonical bundle, set `BUNDLE_TO_REGISTER` to its explicit path and
`REGISTER_MODEL = True`. A NO₂ serving fit is trained separately with LightGBM's
`protocol="serving", fold=None`; it has no LOSO accuracy claim.
Configure `MLFLOW_TRACKING_URI`, `MLFLOW_S3_ENDPOINT_URL`, `AWS_ACCESS_KEY_ID`, and
`AWS_SECRET_ACCESS_KEY` outside the notebook. Do not paste credentials into cells or outputs.
Registration creates a candidate version. Promotion requires its own `PROMOTE_MODEL = True`.

In [ ]:
registered = None
if REGISTER_MODEL:
    from spatial_model.artifacts.bundles import validate_bundle
    from spatial_model.artifacts.registry import register
    if BUNDLE_TO_REGISTER is None:
        raise ValueError("Set BUNDLE_TO_REGISTER to a reviewed canonical serving bundle.")
    candidate = validate_bundle(Path(BUNDLE_TO_REGISTER))
    registered = register(Path(BUNDLE_TO_REGISTER))
    display(registered)
else:
    print("Registration disabled.")

In [ ]:
if PROMOTE_MODEL:
    from spatial_model.artifacts.registry import promote
    if registered is None:
        raise ValueError("Register and review a canonical bundle before enabling promotion.")
    print(promote(candidate["target"], registered["version"]))
else:
    print("Promotion disabled.")